# Description

In this notebook I extract the NorESM2-LM  outputs to assess subsampling. The model outputs required are temperature, salinity, aou, agessc as well as latitude, longitude, depth, year. Model output will be yearly averages and will be organize in a text file, each colum will be one of the variable, each row one data point.



# Import modules

In [1]:
%%time
%load_ext memory_profiler

#___________________________
# basics
import datetime
import os, glob, sys, gc
# import warnings
# warnings.filterwarnings('ignore', '.*invalid value encountered in true_divide.*', )

#___________________________
# To follow computations
from dask.diagnostics import ProgressBar
pbar = ProgressBar(minimum=10)
pbar.register()
#pbar.unregister()

#___________________________
# xarray numpy...
import numpy as np
import xarray as xr
xr.set_options(keep_attrs=True)
import pandas as pd


CPU times: user 531 ms, sys: 996 ms, total: 1.53 s
Wall time: 665 ms


# Starters

In [2]:
%%time
%%memit -c
print('################################')
print('################################')
print(datetime.datetime.now())
print('################################')

dirout = '25-11-28-extract-noresm-for-testing-subsampling/'
if not os.path.isdir(dirout) : os.mkdir(dirout)

dirshared = '/mnt/reef-ns1002k/daco/MY_JUPYTER_NOTEBOOKS/OceanICU/SHARED-DATAS/'

netcdfdir = dirout+'netcdf_files/'
if not os.path.isdir(netcdfdir) : os.mkdir(netcdfdir)

sys.stdout.echo = open(dirout+'stdout.txt', 'w')
sys.stderr.echo = open(dirout+'stderr.txt', 'w')

################################
################################
2025-11-28 19:26:22.239352
################################
peak memory: 257.92 MiB, increment: 86.81 MiB
CPU times: user 31.8 ms, sys: 37.6 ms, total: 69.5 ms
Wall time: 141 ms


# Main parameters

In [3]:
%%time
%%memit -c
print('################################')
print('################################')
print(datetime.datetime.now())
print('################################')

kwopends=dict(use_cftime=True, decode_times=None,
              decode_cf=True, decode_coords=True)
kwopenmfds = dict(combine='by_coords', parallel=True, 
                  use_cftime=True, decode_times=None,
                  decode_cf=True, decode_coords=True)


rename_dict = {
    "x": "i",
    "y": "j",
    "lat": "latitude", 
    "lon": "longitude",
    "nav_lat": "latitude", 
    "nav_lon": "longitude",
    'lev': 'depth', 
    'deptht': 'depth', 
    'olevel': 'depth', 
    "Depth":"depth"
}


################################
################################
2025-11-28 19:26:23.382929
################################
peak memory: 258.51 MiB, increment: 108.30 MiB
CPU times: user 30.4 ms, sys: 9.3 ms, total: 39.7 ms
Wall time: 143 ms


# Define some functions

## Others

In [4]:
def check_and_delete_variable(variable_name):
    # Usage example
    # for vvv in ['data2plot', 'data2process_significant', 'data2process_with_low_significant']: 
    #     check_and_delete_variable(vvv)
    # #
    # gc.collect()    
    if variable_name in globals(): 
        del globals()[variable_name]
        print(f"{variable_name} deleted")
    elif variable_name in locals(): 
        del locals()[variable_name]
        print( f"{variable_name} deleted" )
    else:
        print( f"{variable_name} does not exist")

#

def get_esgf_dataset_filepaths(variable, sourceID, experimentID, 
                               freq='mon', grid='g*', version='latest', 
                               variant='r1i1p1f1',
                               mipera = 'CMIP6', diresgf='/mnt/reef-ns1002k-esgf/', verbose=False, **kwargs): 
    """
    Returns the filepaths of the remote netCDF files corresponding to the specified dataset of the Earth System
    Grid Federation (ESGF) data portal on NIRD.

    Parameters:
    -----------
    variable : str
        Variable to search for on ESGF data portal.
    sourceID : str
        Name of the data source on the ESGF data portal.
    experimentID : str
        Name of the experiment on the ESGF data portal.
    freq : str, optional
        Frequency of the data (default is 'mon').
    grid : str, optional
        Type of grid (default is 'g*').
    version : str, optional
        Version of the data being queried (default is 'latest').
    variant : str, optional
        Label for the variant of the data being queried (default is 'r1i1p1f1').
    mipera : str, optional
        Name of the CMIP era being queried (default is 'CMIP6').
    diresgf : str, optional
        Absolute path to the directory where the data is stored (default is '/mnt/reef-ns1002k-esgf/').
    verbose : bool, optional
        If True, prints the function name at the start and end of execution (default is False).
    **kwargs : dict, optional
        Other key-value arguments to be passed in the function.

    Returns:
    --------
    List[str]
        A list of filepaths corresponding to the specified dataset on the ESGF data portal.

    Example:
    --------
    fp_list = get_esgf_dataset_filepaths('tas', 'CanESM5', 'historical', freq='mon')

    Dependencies:
    -------------
    glob, sys
    """
    import glob, sys
    
    if verbose: print('func: get_esgf_dataset_filepaths')
    
    if experimentID in ['1pctCO2', 'piControl', 'historical', 'abrupt-4xCO2']: zwActivity='CMIP'
    elif experimentID in ['ssp126', 'ssp245', 'ssp585']: zwActivity='ScenarioMIP'
    else: sys.exit('Check experimentID, case not implemented')
    
    if sourceID in ['CESM2', 'CESM2-WACCM']: zwInstitutionID = 'NCAR'
    elif sourceID in ['ACCESS-ESM1-5']: zwInstitutionID = 'CSIRO'
    elif sourceID in ['CNRM-ESM2-1']: zwInstitutionID = 'CNRM-CERFACS'
    elif sourceID in ['CanESM5', 'CanESM5-CanOE']: zwInstitutionID = 'CCCma'
    elif sourceID in ['UKESM1-0-LL']: zwInstitutionID = 'NIMS-KMA'
    elif sourceID in ['GFDL-CM4', 'GFDL-ESM4']: zwInstitutionID = 'NOAA-GFDL'
    elif sourceID in ['IPSL-CM6A-LR', 'IPSL-CM6A-LR-INCA']: zwInstitutionID = 'IPSL'
    elif sourceID in ['MIROC-ES2L']: zwInstitutionID = 'MIROC'
    elif sourceID in ['MPI-ESM1-2-LR', 'ICON-ESM-LR']: zwInstitutionID = 'MPI-M'
    elif sourceID in ['NorESM2-LM']: zwInstitutionID = 'NCC'
    else: sys.exit('Check sourceID, case not implemented')
    
    ocean_list = ['fgco2', 'intpp', 'o2', 'thetao', 'so', 'agessc', 'po4', 'no3', 
                  'dissic', 'talk', 'cfc12', 'cfc11', 'sf6']
    if variable in ocean_list: zwTableID = 'O'+freq
    elif variable in ['areacello']: zwTableID='Ofx'
    elif variable in ['psl']: zwTableID='A'+freq
    else: sys.exit('!!! WARNING !!! Check variable, case not implemented')
        
    zwdname = diresgf + mipera +'/'+ zwActivity +'/'+ \
        zwInstitutionID +'/'+ sourceID +'/'+ \
        experimentID  +'/'+ variant +'/'+ zwTableID +'/'+ \
        variable+'/'+ grid +'/'+ version +'/'
    zwfname = variable +'_'+ zwTableID +'_'+ sourceID +'_'+ \
        experimentID +'_'+ variant +'_'+ grid +'*.nc' 

    if verbose: print('endfunc')
    return glob.glob(zwdname + zwfname)
#
def nan_helper(y):
    """Helper to handle indices and logical indices of NaNs.

    Input:
        - y, 1d numpy array with possible NaNs
    Output:
        - nans, logical indices of NaNs
        - index, a function, with signature indices= index(logical_indices),
          to convert logical indices of NaNs to 'equivalent' indices
    Example:
        >>> # linear interpolation of NaNs
        >>> nans, x= nan_helper(y)
        >>> y[nans]= np.interp(x(nans), x(~nans), y[~nans])
        nb: y[~nans] values of y that are not nans
            x(~nans) indexes of y that are not nans
    """

    return np.isnan(y), lambda z: z.nonzero()[0]
#



## Preparation of data 

In [5]:
def shift_180_lon(zwda, verbose=False): 
    if verbose: print("func: shift_180_lon")
    
    try: 
        if not np.nanmin(zwda['longitude']) < -150: 
            zwda['longitude'] = (zwda['longitude'] + 180) % 360 - 180
            addtxt=str(datetime.datetime.now())+' shift_180_lon to get longitude from -180 to 180'
            try: zwda.attrs['history'] =  addtxt + ' ; '+zwda.attrs['history']
            except: zwda.attrs['history'] =  addtxt             
        #
    except: print('WARNING! longitude likely not shifted')
    return zwda
#

def rename_vars_dims_coords(ds, rename_dict, verbose=False):
    """
    Renames variables, dimensions, and coordinates in an xarray Dataset according to the provided rename dictionary.

    Parameters:
    -----------
    ds : xr.Dataset
        The xarray Dataset to be renamed.
    rename_dict : Dict[str, str]
        Dictionary containing the variable, dimension, or coordinate names to be renamed. 
        The keys represent the original names, and the values represent the new names.
    verbose : bool, optional
        If True, prints the function name at the start and end of execution (default is False).
    
    Returns:
    --------
    xr.Dataset
        A new xarray Dataset with variables, dimensions, and coordinates renamed according to the rename dictionary.
    
    Example:
    --------
    import xarray as xr
    data = {'temp': ([], [0]), 'sali': ([], [1])}
    coords = {'time': [0]}
    ds = xr.Dataset(data, coords)
    renamed_ds = rename_vars_dims_coords(ds, {'temp': 'temperature', 'sali': 'salinity'})

    Dependencies:
    -------------
    xarray
    """
    if verbose: print('func: rename_vars_dims_coords')
    for old_name, new_name in rename_dict.items():
        if (old_name in ds.variables) | (old_name in ds.dims) | (old_name in ds.coords): 
            ds = ds.rename({old_name: new_name})
        #
    if verbose: print('endfunc')
    return ds
#

def split_coords_dimensions(ds, verbose=False):
    """
    Splits the latitude, longitude, and depth dimensions and coordinates of an xarray dataset into separate variables,
    updates their names, and assigns them back to the dataset.

    Parameters:
    -----------
    ds : xr.Dataset
        The xarray Dataset to be updated.
    verbose : bool, optional
        If True, prints the function name at the start and end of execution (default is False).
    
    Returns:
    --------
    xr.Dataset
        A new xarray Dataset with the latitude, longitude, and depth dimensions and coordinates split into separate variables
        and reassigned to the original dataset.
    
    Example:
    --------
    import xarray as xr
    data = {'temp': ([0, 1, 2], [0, 1]), 'sali': ([0, 1, 2], [0, 1])}
    coords = {'latitude': [0, 1, 2], 'longitude': [0, 1], 'depth': [0, 1, 2]}
    ds = xr.Dataset(data, coords)
    updated_ds = split_coords_dimensions(ds)

    Dependencies:
    -------------
    xarray
    """
    if verbose: print('func: split_coords_dimensions')
    new_coords = {}
    new_coords2 = {}
    new_dims = {}
    dim_name_dict = dict(latitude='j', longitude='i', depth='k')
    dimschanged = []
    for name, coord in ds.coords.items():
        if name in ds.dims and name in ["latitude", "longitude", "depth"]:
            new_coords[name + "_coord"] = coord
            new_dims[name] = dim_name_dict[name]
            new_coords2[name + "_coord"] = name
            dimschanged.append(name)
    if verbose: print('endfunc')
    for name in ['k', 'j', 'i']: 
        if name in ds.coords: dimschanged.append(name)
    #
    return ds.assign_coords(new_coords).rename_dims(new_dims).drop_vars(dimschanged).rename(new_coords2)
#


# Extract data

In [ ]:
%%time
%%memit -c
# ca. 3 min
print('################################')
print('################################')
print(datetime.datetime.now())
print('# Extract data')
print('################################')

from scipy.spatial import cKDTree


savename_suff = 'for_testing_subsampling_SO.txt'
savename_pref = 'NorESM2-LM'
file_path = dirout + 'SO_lon_lat_depth_year.txt'
print(f'Read {file_path}...')
zwdf = pd.read_csv(file_path, delimiter=',')

tslice = slice('1982', '2013')


#-----------------------
# Load NorESM data
#-----------------------

print('Load NorESM data...')

#________________
# Get potential temperature

print('    Get potential temperature...')

vesm = 'NorESM2-LM'
simu='historical'
var = 'thetao'

fname = dirshared+vesm+"_"+simu+"_"+var+"_198[2-9].nc"
path_list = glob.glob(fname)
fname = dirshared+vesm+"_"+simu+"_"+var+"_199[0-9].nc"
path_list.extend(glob.glob(fname))
fname = dirshared+vesm+"_"+simu+"_"+var+"_20[0-1][0-9].nc"
path_list.extend(glob.glob(fname))
path_list.sort()

zwds = xr.open_mfdataset(path_list, **kwopenmfds) 
zwda_thetao = zwds[var].sel(year=tslice)

#________________
# Get practical salinity

print('    Get practical salinity...')

vesm = 'NorESM2-LM'
simu='historical'
var = 'so'

fname = dirshared+vesm+"_"+simu+"_"+var+"_198[2-9].nc"
path_list = glob.glob(fname)
fname = dirshared+vesm+"_"+simu+"_"+var+"_199[0-9].nc"
path_list.extend(glob.glob(fname))
fname = dirshared+vesm+"_"+simu+"_"+var+"_20[0-1][0-9].nc"
path_list.extend(glob.glob(fname))
path_list.sort()

zwds = xr.open_mfdataset(path_list, **kwopenmfds) 
zwda_so = zwds[var].sel(year=tslice)

#________________
# Get agessc

print('    Get agessc...')

vesm = 'NorESM2-LM'
simu='historical'
var = 'agessc'

fname = dirshared+vesm+"_"+simu+"_"+var+"_198[2-9].nc"
path_list = glob.glob(fname)
fname = dirshared+vesm+"_"+simu+"_"+var+"_199[0-9].nc"
path_list.extend(glob.glob(fname))
fname = dirshared+vesm+"_"+simu+"_"+var+"_20[0-1][0-9].nc"
path_list.extend(glob.glob(fname))
path_list.sort()

zwds = xr.open_mfdataset(path_list, **kwopenmfds) 
zwda_age = zwds[var].sel(year=tslice)

#________________
# Get aou

print('    Get aou...')

vesm = 'NorESM2-LM'
simu='historical'
var = 'aou'

fname = dirshared+vesm+"_"+simu+"_"+var+"_198[2-9].nc"
path_list = glob.glob(fname)
fname = dirshared+vesm+"_"+simu+"_"+var+"_199[0-9].nc"
path_list.extend(glob.glob(fname))
fname = dirshared+vesm+"_"+simu+"_"+var+"_20[0-1][0-9].nc"
path_list.extend(glob.glob(fname))
path_list.sort()

zwds = xr.open_mfdataset(path_list, **kwopenmfds) 
zwda_aou = zwds[var].sel(year=tslice)

#________________
# Gather in dictionnary

print('    Gather dataarray in a dictionnary')

zwda_dict = {
    'thetao': zwda_thetao,
    'so': zwda_so,
    'agessc': zwda_age,
    'aou': zwda_aou,
}


#-----------------------
# Extract nearest index
#-----------------------

print('Extract nearest index...')

lon_grid, lat_grid = zwda_thetao.longitude.values, zwda_thetao.latitude.values
# Use KDTree for 2D nearest neighbor search on latitude and longitude
lat_lon_tree = cKDTree(np.c_[lat_grid.ravel(), lon_grid.ravel()])

# Function to find the nearest index
def find_nearest(array, value):
    idx = (np.abs(array - value)).argmin()
    return idx
#


rownumber = len(zwdf)
for name, zwda in zwda_dict.items(): 
    
    print(f'    For {name}...')
    
    extraction_dict = {name: [], 'year':[], 'dep':[], 'lat':[], 'lon':[]}
    savename = f'{savename_pref}_{name}_{savename_suff}'
    
    for idx, row in zwdf.iterrows():
        
        if idx%2000==0: 
            print(f'        {str(datetime.datetime.now().strftime("%Y-%m-%d %H:%M"))}, {idx}/{rownumber} rows, i.e. {idx/rownumber*100:.0f}% ...')
        #

        # Find nearest indices
        year_idx = find_nearest(zwda['year'].values, row['year'])
        depth_idx = find_nearest(zwda['depth'].values, row['depth'])

        # Find the nearest lat/lon point in the 2D plane
        _, lat_lon_idx = lat_lon_tree.query([row['lat'], row['lon']])

        # Convert linear index to 2D index
        lat_idx, lon_idx = np.unravel_index(lat_lon_idx, lat_grid.shape)

        # Extract the value from the DataArray
        zwda_extracted = zwda.isel(year=year_idx,
                                 k=depth_idx,
                                 j=lat_idx,
                                 i=lon_idx)
        extraction_dict[name].append(zwda_extracted.values)
        extraction_dict['year'].append(zwda_extracted.year.values)
        extraction_dict['dep'].append(zwda_extracted.depth.values)
        extraction_dict['lat'].append(zwda_extracted.latitude.values)
        extraction_dict['lon'].append(zwda_extracted.longitude.values)
        
    #   
    print('    Save dataframe...')
    df2save = pd.DataFrame(extraction_dict)
    df2save.to_csv(dirout+savename, sep=',', index=False)
    print(f'    Done: {dirout+savename}')
#

print('################################')
print('Done', datetime.datetime.now())
print('################################')
print('################################')


################################
################################
2025-11-30 12:30:26.695131
# Extract data
################################
Read 25-11-28-extract-noresm-for-testing-subsampling/SO_lon_lat_depth_year.txt...
Load NorESM data...
    Get potential temperature...
    Get practical salinity...
    Get agessc...
    Get aou...
    Gather dataarray in a dictionnary
Extract nearest index...
    For thetao...
        2025-11-30 12:31, 0/38121 rows, i.e. 0% ...
        2025-11-30 12:50, 2000/38121 rows, i.e. 5% ...
        2025-11-30 13:10, 4000/38121 rows, i.e. 10% ...
        2025-11-30 13:30, 6000/38121 rows, i.e. 16% ...
        2025-11-30 13:50, 8000/38121 rows, i.e. 21% ...
        2025-11-30 14:09, 10000/38121 rows, i.e. 26% ...
        2025-11-30 14:28, 12000/38121 rows, i.e. 31% ...
        2025-11-30 14:47, 14000/38121 rows, i.e. 37% ...
